<a href="https://colab.research.google.com/github/ichaxpro/244107020023-Machine-Learning/blob/master/JS09/JS09_Lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Conduct experiments on different distance metrics, 1,000 versus 1,000,000 data points, 2D versus 5D data, for the algorithms: ANNOY, FAISS, and HNSW

In [ ]:
pip install numpy pandas annoy faiss-cpu hnswlib

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import numpy as np
import pandas as pd
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

# Konfigurasi eksperimen: (Jumlah Data, Dimensi)
configs = [
    (1000, 2),
    (1000, 5),
    (1000000, 2),
    (1000000, 5)
]

# Jumlah query yang akan dites (mencari 5 tetangga terdekat)
num_queries = 100
k = 5
metric = 'l2'

results = []

for num_data, dim in configs:
    print(f"Menjalankan eksperimen untuk {num_data} data, {dim}D...")

    # Generate data random dan query random
    data = np.random.random((num_data, dim)).astype('float32')
    queries = np.random.random((num_queries, dim)).astype('float32')

    row_result = {"Number of Data/Dimensions": f"{num_data:,}/{dim}D"}

    # ---------------------------------------------------------
    # 1. ANNOY
    # ---------------------------------------------------------
    annoy_index = AnnoyIndex(dim, 'euclidean')
    for i in range(num_data):
        annoy_index.add_item(i, data[i])
    annoy_index.build(10) # 10 trees

    start_time = time.time()
    for i in range(num_queries):
        annoy_index.get_nns_by_vector(queries[i], k)
    annoy_time = time.time() - start_time
    row_result["ANNOY"] = f"{annoy_time:.5f} s"

    # ---------------------------------------------------------
    # 2. FAISS (menggunakan IndexFlatL2 untuk baseline/exact search)
    # ---------------------------------------------------------
    faiss_index = faiss.IndexFlatL2(dim)
    faiss_index.add(data)

    start_time = time.time()
    faiss_index.search(queries, k)
    faiss_time = time.time() - start_time
    row_result["FAISS"] = f"{faiss_time:.5f} s"

    # ---------------------------------------------------------
    # 3. HNSW (hnswlib)
    # ---------------------------------------------------------
    hnsw_index = hnswlib.Index(space=metric, dim=dim)
    hnsw_index.init_index(max_elements=num_data, ef_construction=200, M=16)
    hnsw_index.add_items(data)
    hnsw_index.set_ef(50)

    start_time = time.time()
    hnsw_index.knn_query(queries, k=k)
    hnsw_time = time.time() - start_time
    row_result["HNSW"] = f"{hnsw_time:.5f} s"

    results.append(row_result)

# Menampilkan hasil dalam bentuk tabel
df_results = pd.DataFrame(results)
print("\n=== HASIL PERFORMANSI (Waktu untuk 100 Query) ===")
print(df_results.to_markdown(index=False))

C:\Users\icha\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Menjalankan eksperimen untuk 1000 data, 2D...
Menjalankan eksperimen untuk 1000 data, 5D...
Menjalankan eksperimen untuk 1000000 data, 2D...
Menjalankan eksperimen untuk 1000000 data, 5D...

=== HASIL PERFORMANSI (Waktu untuk 100 Query) ===
| Number of Data/Dimensions   | ANNOY     | FAISS     | HNSW      |
|:----------------------------|:----------|:----------|:----------|
| 1,000/2D                    | 0.00351 s | 0.00269 s | 0.00345 s |
| 1,000/5D                    | 0.00290 s | 0.00040 s | 0.00372 s |
| 1,000,000/2D                | 0.00612 s | 0.23739 s | 0.00325 s |
| 1,000,000/5D                | 0.00514 s | 0.13119 s | 0.00405 s |
